In [1]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import numpy as np
import pandas as pd

In [2]:
df = pd.read_parquet("data/processed/model_table.parquet")
y = df["default_flag"]
x = df.drop(columns=["id", "default_flag", "issue_year", "issue_date", "funded_amnt", "annual_inc"])
x = pd.get_dummies(x, columns=["home_ownership", "verification_status", "purpose"], drop_first=True, dtype=int)

print(f"Shape of x: {x.shape}")
print(f"Mean of y: {y.mean()}")

Shape of x: (545779, 80)
Mean of y: 0.14068331687367963


In [3]:
train_mask = df["issue_year"] <= 2014
x_train = x.loc[train_mask]
y_train = y.loc[train_mask]
x_test = x.loc[~train_mask]
y_test = y.loc[~train_mask]

print(f"Rows in train set: {y_train.shape[0]}, Rows in test set: {y_test.shape[0]}")
print(f"Mean of y_train: {y_train.mean()}, Mean of y_test: {y_test.mean()}")

Rows in train set: 262992, Rows in test set: 282787
Mean of y_train: 0.1319165601995498, Mean of y_test: 0.14883640337073487


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import roc_auc_score 

scaler = StandardScaler()
x_train_s = scaler.fit_transform(x_train)
x_test_s = scaler.transform(x_test)

model = LogisticRegression(max_iter=1000)
model.fit(x_train_s, y_train)

pd_test = model.predict_proba(x_test_s)[:, 1]

print(f"roc_auc_score: {roc_auc_score(y_test, pd_test)}")
print(f"roc_auc_score (train): {roc_auc_score(y_train, model.predict_proba(x_train_s)[:, 1])}")
print(f"mean of pd_test: {pd_test.mean()}")

roc_auc_score: 0.683909518394715
roc_auc_score (train): 0.6773873853202709
mean of pd_test: 0.13617472692738952


In [5]:
import duckdb
conn = duckdb.connect('data/loans.duckdb', read_only=True)

grades = conn.sql("SELECT id, sub_grade FROM loans").df()
grades_merged = df.loc[~train_mask].merge(grades, on="id", how="left")
print(f"Rows in merged: {grades_merged.shape[0]}")
print(f"Example sub_grade: {grades_merged['sub_grade'].iloc[100]}")

grade_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
grades_merged["sub_grade_num"] = grades_merged["sub_grade"].apply(lambda s: grade_map.get(s[0], 0) * 5 + int(s[1]))
print(f"ROC AUC Score: {roc_auc_score(grades_merged['default_flag'], grades_merged['sub_grade_num'])}")
print(f"5 Example sub_grade_num: {grades_merged['sub_grade_num'].iloc[100]}, {grades_merged['sub_grade_num'].iloc[101]}, {grades_merged['sub_grade_num'].iloc[102]}, {grades_merged['sub_grade_num'].iloc[103]}, {grades_merged['sub_grade_num'].iloc[104]}")

Rows in merged: 282787
Example sub_grade: B2
ROC AUC Score: 0.6786417383122245
5 Example sub_grade_num: 7, 4, 18, 8, 10


In [ ]:
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_test, pd_test)
ks_lr = (tpr-fpr).max()
print(f"KS Statistic (Logistic Regression): {ks_lr}")

fpr, tpr, thresholds = roc_curve(grades_merged['default_flag'], grades_merged['sub_grade_num'])
ks_grade = (tpr-fpr).max()
print(f"KS Statistic (Sub Grade Benchmark): {ks_grade}")

df2 = pd.DataFrame({"y_test": y_test, "pd_test": pd_test})
decile = pd.qcut(pd_test, 10, labels=False)
calib_table = df2.groupby(decile).agg({"y_test": ["count", "mean"], "pd_test": ["mean"]}).sort_index(ascending=False)
print(calib_table)

KS Statistic (Logistic Regression): 0.2672289009347664
KS Statistic (Sub Grade Benchmark): 0.26194575341328713
  y_test             pd_test
   count      mean      mean
9  28279  0.319460  0.308101
8  28279  0.237986  0.218123
7  28278  0.201358  0.179522
6  28279  0.172425  0.152488
5  28278  0.147182  0.130640
4  28279  0.124403  0.111545
3  28279  0.100746  0.093895
2  28278  0.086074  0.076668
1  28279  0.062838  0.058137
0  28279  0.035892  0.032625


In [ ]:
import statsmodels.api as sm
x_train_df = pd.DataFrame(x_train_s, columns=x_train.columns)
x_train_df = sm.add_constant(x_train_df)
logit = sm.Logit(y_train.values, x_train_df).fit()

params_table = pd.DataFrame({"coefficients": logit.params, "p_values": logit.pvalues}).sort_values(by="coefficients", key=abs, ascending=False)
print(f"Top 15 coefficients:\n{params_table.head(15)}")

AttributeError: module 'statsmodels.api' has no attribute 'add_constraint'